# 第 3 步：高效变体模型训练

本 notebook 完成以下任务：
1. 实现并训练 Informer 模型
2. 实现并训练 Autoformer 模型
3. 实现并训练 PatchTST 模型
4. 与基线模型对比

**核心创新**：
- **Informer**：ProbSparse 注意力 O(L log L)
- **Autoformer**：序列分解 + Auto-Correlation
- **PatchTST**：Patch 切分 + Channel Independence

## 1. 环境检查与导入

In [ ]:
import sys
import os

# 添加项目根目录到路径
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
from models import InformerModel, AutoformerModel, PatchTSTModel, TimeSeriesDataset, Trainer
from torch.utils.data import DataLoader

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'设备: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 2. 数据集配置

In [ ]:
# 数据集配置
dataset=["ETTh1","ETTM1","ECL"]
HORIZON_LIST=[24,48,96,168,336]

DATASET = dataset[0]  # 可选: ETTh1, ETTm1, ECL
HORIZON = HORIZON_LIST[0]     # 预测步长: 24, 48, 96, 168, 336
BATCH_SIZE = 32
DATA_DIR = os.path.join(ROOT, 'data', 'processed')

print(f'数据集: {DATASET}')
print(f'预测步长: {HORIZON}')
print(f'批次大小: {BATCH_SIZE}')

## 3. 加载数据

In [ ]:
# 创建数据加载器
train_dataset = TimeSeriesDataset(DATA_DIR, DATASET, HORIZON, 'train')
val_dataset = TimeSeriesDataset(DATA_DIR, DATASET, HORIZON, 'val')
test_dataset = TimeSeriesDataset(DATA_DIR, DATASET, HORIZON, 'test')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

input_size = train_dataset.input_size
target_idx = train_dataset.target_idx

print(f'输入特征数: {input_size}')
print(f'目标列索引: {target_idx}')
print(f'训练样本: {len(train_dataset)}')
print(f'验证样本: {len(val_dataset)}')
print(f'测试样本: {len(test_dataset)}')

## 4. 训练 Informer 模型

In [ ]:
# 创建 Informer 模型
informer_model = InformerModel(
    input_size=input_size,
    d_model=128,
    n_heads=8,
    n_encoder_layers=3,
    n_decoder_layers=2,
    d_ff=256,
    factor=5,  # ProbSparse 的 Top-K 因子
    dropout=0.1,
    horizon=HORIZON
)

print(f'Informer 模型参数量: {sum(p.numel() for p in informer_model.parameters()):,}')

In [ ]:
# 训练 Informer
informer_trainer = Trainer(informer_model, device='cuda', lr=1e-4)
informer_history = informer_trainer.train(
    train_loader, val_loader,
    epochs=100,
    patience=15,
    save_dir=os.path.join(ROOT, 'checkpoints'),
    model_name=f'Informer_{DATASET}_h{HORIZON}',
    log_dir=os.path.join(ROOT, 'runs/Informer')
)

In [ ]:
# 绘制 Informer 训练曲线
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 损失曲线
axes[0].plot(informer_history['train_losses'], label='训练损失')
axes[0].plot(informer_history['val_losses'], label='验证损失')
axes[0].set_title('Informer 损失曲线')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True)

# R² 曲线
axes[1].plot(informer_history['train_r2'], label='训练 R²')
axes[1].plot(informer_history['val_r2'], label='验证 R²')
axes[1].set_title('Informer R² 曲线')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].legend()
axes[1].grid(True)

# 测试指标
informer_preds, informer_targets = informer_trainer.predict(test_loader)
informer_metrics = informer_trainer.compute_metrics(informer_preds, informer_targets, target_idx)

metrics_names = ['MSE', 'MAE', 'MAPE', 'R2']
values = [informer_metrics['MSE'], informer_metrics['MAE'], informer_metrics['MAPE'], informer_metrics['R2']]
axes[2].bar(metrics_names, values, color=['blue', 'orange', 'green', 'red'])
axes[2].set_title('Informer 测试指标')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f'Informer 测试结果:')
print(f'  MSE: {informer_metrics["MSE"]:.6f}')
print(f'  MAE: {informer_metrics["MAE"]:.6f}')
print(f'  MAPE: {informer_metrics["MAPE"]:.2f}%')
print(f'  R²: {informer_metrics["R2"]:.4f}')

## 5. 训练 Autoformer 模型

In [ ]:
# 创建 Autoformer 模型
autoformer_model = AutoformerModel(
    input_size=input_size,
    d_model=64,        # 从 128 减到 64
    n_heads=4,         # 从 8 减到 4
    n_encoder_layers=2, # 从 3 减到 2
    n_decoder_layers=1, # 从 2 减到 1
    d_ff=128,          # 从 256 减到 128
    factor=3,  # Auto-Correlation Top-K
    dropout=0.1,
    horizon=HORIZON,
    kernel_size=25  # 移动平均窗口
)

print(f'Autoformer 模型参数量: {sum(p.numel() for p in autoformer_model.parameters()):,}')

In [19]:
# 训练 Autoformer
autoformer_trainer = Trainer(autoformer_model, device='cuda', lr=1e-4)
autoformer_history = autoformer_trainer.train(
    train_loader, val_loader,
    epochs=100,
    patience=15,
    save_dir=os.path.join(ROOT, 'checkpoints'),
    model_name=f'Autoformer_{DATASET}_h{HORIZON}',
    log_dir=os.path.join(ROOT, 'runs/Autoformer')
)

KeyboardInterrupt: 

In [ ]:
# 绘制 Autoformer 训练曲线
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 损失曲线
axes[0].plot(autoformer_history['train_losses'], label='训练损失')
axes[0].plot(autoformer_history['val_losses'], label='验证损失')
axes[0].set_title('Autoformer 损失曲线')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True)

# R² 曲线
axes[1].plot(autoformer_history['train_r2'], label='训练 R²')
axes[1].plot(autoformer_history['val_r2'], label='验证 R²')
axes[1].set_title('Autoformer R² 曲线')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].legend()
axes[1].grid(True)

# 测试指标
autoformer_preds, autoformer_targets = autoformer_trainer.predict(test_loader)
autoformer_metrics = autoformer_trainer.compute_metrics(autoformer_preds, autoformer_targets, target_idx)

values = [autoformer_metrics['MSE'], autoformer_metrics['MAE'], autoformer_metrics['MAPE'], autoformer_metrics['R2']]
axes[2].bar(metrics_names, values, color=['blue', 'orange', 'green', 'red'])
axes[2].set_title('Autoformer 测试指标')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f'Autoformer 测试结果:')
print(f'  MSE: {autoformer_metrics["MSE"]:.6f}')
print(f'  MAE: {autoformer_metrics["MAE"]:.6f}')
print(f'  MAPE: {autoformer_metrics["MAPE"]:.2f}%')
print(f'  R²: {autoformer_metrics["R2"]:.4f}')

## 6. 训练 PatchTST 模型

In [ ]:
# 创建 PatchTST 模型
patchtst_model = PatchTSTModel(
    input_size=input_size,
    d_model=128,
    n_heads=8,
    n_layers=3,
    d_ff=256,
    patch_len=16,  # 每个 patch 的长度
    stride=8,      # patch 的步长
    dropout=0.1,
    horizon=HORIZON
)

print(f'PatchTST 模型参数量: {sum(p.numel() for p in patchtst_model.parameters()):,}')

In [ ]:
# 训练 PatchTST
patchtst_trainer = Trainer(patchtst_model, device='cuda', lr=1e-4)
patchtst_history = patchtst_trainer.train(
    train_loader, val_loader,
    epochs=100,
    patience=15,
    save_dir=os.path.join(ROOT, 'checkpoints'),
    model_name=f'PatchTST_{DATASET}_h{HORIZON}',
    log_dir=os.path.join(ROOT, 'runs/PatchTST')
)

In [ ]:
# 绘制 PatchTST 训练曲线
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 损失曲线
axes[0].plot(patchtst_history['train_losses'], label='训练损失')
axes[0].plot(patchtst_history['val_losses'], label='验证损失')
axes[0].set_title('PatchTST 损失曲线')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True)

# R² 曲线
axes[1].plot(patchtst_history['train_r2'], label='训练 R²')
axes[1].plot(patchtst_history['val_r2'], label='验证 R²')
axes[1].set_title('PatchTST R² 曲线')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].legend()
axes[1].grid(True)

# 测试指标
patchtst_preds, patchtst_targets = patchtst_trainer.predict(test_loader)
patchtst_metrics = patchtst_trainer.compute_metrics(patchtst_preds, patchtst_targets, target_idx)

values = [patchtst_metrics['MSE'], patchtst_metrics['MAE'], patchtst_metrics['MAPE'], patchtst_metrics['R2']]
axes[2].bar(metrics_names, values, color=['blue', 'orange', 'green', 'red'])
axes[2].set_title('PatchTST 测试指标')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f'PatchTST 测试结果:')
print(f'  MSE: {patchtst_metrics["MSE"]:.6f}')
print(f'  MAE: {patchtst_metrics["MAE"]:.6f}')
print(f'  MAPE: {patchtst_metrics["MAPE"]:.2f}%')
print(f'  R²: {patchtst_metrics["R2"]:.4f}')

## 7. 加载基线模型结果进行对比

In [ ]:
# 加载基线模型结果
results_dir = os.path.join(ROOT, 'results')
baseline_results_path = os.path.join(results_dir, f'{DATASET}_h{HORIZON}_results.npy')

if os.path.exists(baseline_results_path):
    baseline_results = np.load(baseline_results_path, allow_pickle=True).item()
    lstm_metrics = baseline_results['lstm']
    transformer_metrics = baseline_results['transformer']
    print('已加载基线模型结果')
else:
    print('未找到基线模型结果，请先运行 train_baseline.ipynb')
    lstm_metrics = None
    transformer_metrics = None

## 8. 模型对比

In [ ]:
# 模型对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 验证损失对比
axes[0].plot(informer_history['val_losses'], label='Informer')
axes[0].plot(autoformer_history['val_losses'], label='Autoformer')
axes[0].plot(patchtst_history['val_losses'], label='PatchTST')
axes[0].set_title('验证损失对比')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True)

# R² 对比
axes[1].plot(informer_history['val_r2'], label='Informer')
axes[1].plot(autoformer_history['val_r2'], label='Autoformer')
axes[1].plot(patchtst_history['val_r2'], label='PatchTST')
axes[1].set_title('验证 R² 对比')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].legend()
axes[1].grid(True)

# 测试指标对比
metrics_names = ['MSE', 'MAE', 'MAPE', 'R2']
x = np.arange(len(metrics_names))
width = 0.2

if lstm_metrics:
    axes[2].bar(x - 1.5*width, [lstm_metrics[m] for m in metrics_names], width, label='LSTM')
    axes[2].bar(x - 0.5*width, [transformer_metrics[m] for m in metrics_names], width, label='Transformer')
axes[2].bar(x + 0.5*width, [informer_metrics[m] for m in metrics_names], width, label='Informer')
axes[2].bar(x + 1.5*width, [autoformer_metrics[m] for m in metrics_names], width, label='Autoformer')
axes[2].bar(x + 2.5*width, [patchtst_metrics[m] for m in metrics_names], width, label='PatchTST')
axes[2].set_title('测试指标对比')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_names)
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 打印对比表
print('\n' + '=' * 80)
print('模型对比')
print('=' * 80)
print(f'{"指标":<10} {"LSTM":<15} {"Transformer":<15} {"Informer":<15} {"Autoformer":<15} {"PatchTST":<15}')
print('-' * 80)
for m in metrics_names:
    lstm_val = f'{lstm_metrics[m]:.6f}' if lstm_metrics else 'N/A'
    trans_val = f'{transformer_metrics[m]:.6f}' if transformer_metrics else 'N/A'
    print(f'{m:<10} {lstm_val:<15} {trans_val:<15} {informer_metrics[m]:<15.6f} {autoformer_metrics[m]:<15.6f} {patchtst_metrics[m]:<15.6f}')
print('=' * 80)

## 9. 预测可视化

In [ ]:
# 选择一个样本进行可视化
sample_idx = 0
n_steps = min(200, HORIZON)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Informer 预测
axes[0].plot(informer_targets[sample_idx, :n_steps, target_idx], label='真实值', linewidth=2)
axes[0].plot(informer_preds[sample_idx, :n_steps, target_idx], label='Informer 预测', linestyle='--')
axes[0].set_title(f'Informer 预测 vs 真实值 (样本 {sample_idx})')
axes[0].set_xlabel('时间步')
axes[0].set_ylabel('归一化值')
axes[0].legend()
axes[0].grid(True)

# Autoformer 预测
axes[1].plot(autoformer_targets[sample_idx, :n_steps, target_idx], label='真实值', linewidth=2)
axes[1].plot(autoformer_preds[sample_idx, :n_steps, target_idx], label='Autoformer 预测', linestyle='--')
axes[1].set_title(f'Autoformer 预测 vs 真实值 (样本 {sample_idx})')
axes[1].set_xlabel('时间步')
axes[1].set_ylabel('归一化值')
axes[1].legend()
axes[1].grid(True)

# PatchTST 预测
axes[2].plot(patchtst_targets[sample_idx, :n_steps, target_idx], label='真实值', linewidth=2)
axes[2].plot(patchtst_preds[sample_idx, :n_steps, target_idx], label='PatchTST 预测', linestyle='--')
axes[2].set_title(f'PatchTST 预测 vs 真实值 (样本 {sample_idx})')
axes[2].set_xlabel('时间步')
axes[2].set_ylabel('归一化值')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 10. 保存结果

In [ ]:
# 保存变体模型结果
variant_results = {
    'dataset': DATASET,
    'horizon': HORIZON,
    'informer': informer_metrics,
    'autoformer': autoformer_metrics,
    'patchtst': patchtst_metrics
}

np.save(os.path.join(results_dir, f'{DATASET}_h{HORIZON}_variant_results.npy'), variant_results)
print(f'变体模型结果已保存至: {results_dir}/{DATASET}_h{HORIZON}_variant_results.npy')

In [ ]:
# 打印最终结果
print('\n' + '=' * 60)
print('高效变体模型训练完成')
print('=' * 60)
print(f'数据集: {DATASET}')
print(f'预测步长: {HORIZON}')
print(f'\nInformer 最佳验证损失: {informer_history["best_val_loss"]:.6f}')
print(f'Informer 最佳验证 R²: {informer_history["best_val_r2"]:.4f}')
print(f'\nAutoformer 最佳验证损失: {autoformer_history["best_val_loss"]:.6f}')
print(f'Autoformer 最佳验证 R²: {autoformer_history["best_val_r2"]:.4f}')
print(f'\nPatchTST 最佳验证损失: {patchtst_history["best_val_loss"]:.6f}')
print(f'PatchTST 最佳验证 R²: {patchtst_history["best_val_r2"]:.4f}')
print(f'\n模型已保存至: checkpoints/')
print('=' * 60)

## 11. 下一步

1. **多数据集实验**：在 ETTm1 和 ECL 上运行
2. **多步长实验**：测试 horizon = 48, 96, 168, 336
3. **消融实验**：验证各组件的有效性
4. **深入分析**：长期预测性能衰减、误差累积等